In [ ]:
import os
import re
import warnings
import random
from collections import defaultdict
from typing import Dict, List, Tuple

import torch
import torch.nn.functional as F
import numpy as np
import numpy as np
import torch
from tqdm.notebook import tqdm
from transformers import GPT2LMHeadModel, GPT2Tokenizer

warnings.filterwarnings('ignore')

In [2]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


seed_everything(42)

## Задание

1) Реализовать методы `greedy_sampling` и `generate` (1 балл)
2) Реализовать метод `random_sampling` и поддержать его в `generate` (1 балл)
3) Реализовать метод `_beam_search_generate` и поддержать его в `generate` (2 балла)
4) Реализовать методы `apply_top_p`, `apply_top_k`, `apply_temperature` и поддержать их в `generate` (1 балл)  
Все методы необходимо реализовать через векторные операции в torch/numpy везде где это возможно

In [ ]:
class Model:
    def __init__(self, model_name: str = 'gpt2'):
        self.model = GPT2LMHeadModel.from_pretrained(model_name)
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.vocab_size = self.tokenizer.vocab_size

    def greedy_sampling(self, logits: torch.Tensor) -> int:

        return torch.argmax(logits)

    def random_sampling(self, logits: torch.Tensor) -> int:

        probs = torch.softmax(logits, dim=-1)

        sampled_token = torch.multinomial(probs, num_samples=1)

        return sampled_token.item()

    def _beam_search_generate(
        self,
        prompt: str,
        max_length: int,
        num_beams: int
    ) -> str:

        input_ids = self.tokenizer.encode(prompt, return_tensors='pt')

        beams = [(input_ids[0], 0.0)]

        for step in range(max_length):
            all_candidates = []

            for sequence, score in beams:

                if sequence[-1] == self.tokenizer.eos_token_id:
                    all_candidates.append((sequence, score))
                    continue

                with torch.no_grad():
                    outputs = self.model(sequence.unsqueeze(0))
                    next_token_logits = outputs.logits[0, -1, :]
                    next_probs = torch.log_softmax(next_token_logits, dim=-1)

                top_probs, top_tokens = torch.topk(next_probs, num_beams)

                for i in range(num_beams):
                    new_token = top_tokens[i].unsqueeze(0)
                    new_sequence = torch.cat([sequence, new_token])
                    new_score = score + top_probs[i].item()
                    all_candidates.append((new_sequence, new_score))

            all_candidates.sort(key=lambda x: x[1], reverse=True)
            beams = all_candidates[:num_beams]

            if all(seq[-1] == self.tokenizer.eos_token_id for seq, score in beams):
                break

        best_sequence, best_score = beams[0]
        return self.tokenizer.decode(best_sequence, skip_special_tokens=True)

    def apply_temperature(self, logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:

        return logits / temperature

    def _apply_top_p(self, logits: torch.Tensor, top_p: float = 1.0) -> torch.Tensor:

        if top_p >= 1.0:
            return logits

        probs = torch.softmax(logits, dim=-1)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        mask = cumulative_probs <= top_p

        expanded_mask = torch.zeros_like(logits, dtype=torch.bool)
        expanded_mask.scatter_(-1, sorted_indices, mask)

        filtered_logits = logits.masked_fill(~expanded_mask, float(0))

        return filtered_logits

    def _apply_top_k(self, logits: torch.Tensor, top_k: float = 0.0) -> torch.Tensor:

        if top_k == 0 or top_k >= logits.size(-1):
            return logits

        values, indices = torch.topk(logits, top_k, dim=-1)
        filtered_logits = torch.full_like(logits, float(0))

        filtered_logits.scatter_(-1, indices, values)

        return filtered_logits

    def generate(
        self,
        prompt: str,
        max_length: int = 50,
        strategy: str = 'greedy',
        temperature: float = 1.0,
        top_k: int = 0,
        top_p: float = 1.0,
        num_beams: int = 3
    ) -> str:

        if strategy == 'beam_search':
            return self._beam_search_generate(prompt, max_length, num_beams)

        input_ids = self.tokenizer.encode(prompt, return_tensors='pt')

        for step in range(max_length - len(input_ids[0])):
            with torch.no_grad():
                outputs = self.model(input_ids)
                next_token_logits = outputs.logits[:, -1, :]

                if strategy == 'random':

                    next_token_logits = self.apply_temperature(
                        next_token_logits, temperature)

                    next_token_logits = self._apply_top_k(
                        next_token_logits, top_k)
                    next_token_logits = self._apply_top_p(
                        next_token_logits, top_p)

                    next_token_id = self.random_sampling(next_token_logits)
                else:
                    next_token_id = self.greedy_sampling(next_token_logits)

            input_ids = torch.cat([
                input_ids,
                torch.tensor([[next_token_id]], dtype=torch.long)
            ], dim=-1)

            if next_token_id == self.tokenizer.eos_token_id:
                break

        return self.tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [ ]:
model = Model('gpt2')

prompt = 'the cat sat on the mat'

result_greedy = model.generate(
    prompt=prompt,
    max_length=20,
    strategy='greedy'
)
print('GREEDY')
print(f'Input: {prompt}')
print(f'Output: {result_greedy}')
print()

result_random = model.generate(
    prompt=prompt,
    max_length=20,
    strategy='random',
    temperature=1.0
)
print('RANDOM, T = 1')
print(f'Input: {prompt}')
print(f'Output: {result_random}')
print()

result_random_cool = model.generate(
    prompt=prompt,
    max_length=20,
    strategy='random',
    temperature=0.8
)
print('RANDOM, T = 0.8')
print(f'Input: {prompt}')
print(f'Output: {result_random_cool}')
print()

result_topk = model.generate(
    prompt=prompt,
    max_length=20,
    strategy='random',
    top_k=10
)
print('RANDOM, K = 10')
print(f'Input: {prompt}')
print(f'Output: {result_topk}')
print()

result_topp = model.generate(
    prompt=prompt,
    max_length=20,
    strategy='random',
    top_p=0.9
)
print('RANDOM, P = 0.9')
print(f'Input: {prompt}')
print(f'Output: {result_topp}')
print()

result_all = model.generate(
    prompt=prompt,
    max_length=20,
    strategy='random',
    temperature=0.7,
    top_k=20,
    top_p=0.95
)
print('RANDOM, T = 0.7, K = 20, P = 0.95')
print(f'Input: {prompt}')
print(f'Output: {result_all}')

GREEDY
Input: the cat sat on the mat
Output: the cat sat on the mat, and the cat sat on the mat, and the cat sat on

RANDOM, T = 1
Input: the cat sat on the mat
Output: the cat sat on the mat all night holding the head of a golden crocodile. Memorial gardens

RANDOM, T = 0.8
Input: the cat sat on the mat
Output: the cat sat on the mat on the right," says Maron, who isn't sure if he

RANDOM, K = 10
Input: the cat sat on the mat
Output: the cat sat on the mat Speedlas Gamer Nothingoperation Spider botchedBA constitutionally describe checkpoints REC figuring frequency

RANDOM, P = 0.9
Input: the cat sat on the mat
Output: the cat sat on the mat More seizing rob sens shitty slicorenJO scrape outcome????????Friday huhoring

RANDOM, T = 0.7, K = 20, P = 0.95
Input: the cat sat on the mat
Output: the cat sat on the matsector creek claimant deleting CISResult secondaryixie stacks� Maher Developer1100 though


In [ ]:
result_beam = model.generate(
    prompt=prompt,
    max_length=20,
    strategy='beam_search',
    num_beams=3
)
print(f'Input: {prompt}')
print(f'Output: {result_beam}')

Input: the cat sat on the mat
Output: the cat sat on the mat.

"I don't know what you're talking about," she said.

"
